In [13]:
# Install the required packages
%pip install --quiet duckdb
%pip install --quiet jupysql
%pip install --quiet duckdb-engine
%pip install --quiet pandas
%pip install --quiet matplotlib
%pip install --quiet requests
%pip install --quiet geopandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Configure the jupyter
import pandas as pd
pd.set_option('display.max_colwidth', None)
# Import jupysql Jupyter extension to create SQL cells
%load_ext sql
%config SqlMagic.autopandas = False
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%config SqlMagic.named_parameters="enabled"

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [33]:
# Configure duckdb
%sql duckdb:///:memory:transmission
%sql load spatial

Deploy Streamlit apps for free on Ploomber Cloud! Learn more: https://ploomber.io/s/signup


Success


In [34]:
%%sql /* A method that returns the direction of a span using the start and end point in degrees*/
create or replace macro directionOfSpan(span) AS 
(degrees(atan2(ST_Y(st_EndPoint(span)) - ST_Y(st_StartPoint(span)), ST_X(st_EndPoint(span)) - ST_X(st_StartPoint(span)))) + 270) % 360;

Count


In [41]:
%%sql /* #Query 1 Take all transmissions spans and split them into individual components */
create or replace view spans as
with allTransmissions as
(
       -- Turn everything into spanstrings
       from st_read('Transmission_Overhead_Powerlines_WP_032_WA_GDA2020_Public_Secure_Shapefile/Transmission_Overhead_Powerlines_WP_032.shp')
       select
              case when ST_GeometryType(geom) = 'MULTISTRING' then unnest(ST_DUMP(geom)).geom
                     else geom end as geom,
              * exclude (geom)
)
from allTransmissions, range(1,800)
select
       -- The geometry are the individual line spans
       st_makeline(st_pointN(geom, range::int), st_pointN(geom, range::int + 1)) as span,
       -- Features
       directionOfSpan(span) as "direction",
       line_name as "line name",
       range as "index",
       kv "capacity", 
       -- Foreign key to the Era5 files
       round(st_x(ST_StartPoint(span))*4)/4 as "weather longitude",
       round(st_y(ST_StartPoint(span))*4)/4 as "weather latitude",
where span is not null;

select st_astext(span), * exclude(span) from spans limit 5;

st_astext(span),direction,line name,index,capacity,weather longitude,weather latitude
"LINESTRING (115.11902463000001 -28.890160304999938, 115.11922049500004 -28.89039756099993)",219.5411339597684,MGA-TS 81,1,132,115.0,-29.0
"LINESTRING (115.11922049500004 -28.89039756099993, 115.1198443720001 -28.89024556199996)",283.69256038329115,MGA-TS 81,2,132,115.0,-29.0
"LINESTRING (115.1198443720001 -28.89024556199996, 115.12114436500008 -28.891128513999945)",235.81575084149637,MGA-TS 81,3,132,115.0,-29.0
"LINESTRING (115.12114436500008 -28.891128513999945, 115.12244247500007 -28.892020540999965)",235.50417565991935,MGA-TS 81,4,132,115.0,-29.0
"LINESTRING (115.12244247500007 -28.892020540999965, 115.12371744300003 -28.892883539999957)",235.9067559259436,MGA-TS 81,5,132,115.0,-29.0


In [38]:
# Visualise the data
%sql copy spans to 'maps/1. Spans.geojson' with (format GDAL, driver 'geojson');

%sql select concat('https://www.google.com/maps/@', st_y(st_StartPoint(span)), ',', st_x(st_StartPoint(span)), ',1004m/data=!3m1!1e3?entry=ttu') as link from spans where "line name" = 'MOR-TS 81' and index=99;

link
"https://www.google.com/maps/@-30.367616589999955,116.00897670300003,1004m/data=!3m1!1e3?entry=ttu"


In [42]:
%%sql /* Query #2 Get the weather and the KPI */
create or replace view weather as
from 'Era5/era5_australia_20**.parquet'
select -- Geometry, key    
       latitude,
       longitude,
       -- Time, key
       date_part('Month', time) as month,
       date_part('hour', time)::int as hour,
       -- Features
       avg(t2m) - 273.15 as "avg temperature", -- switch from Kelvin
       avg(u10) as "avg u10",
       avg(v10) as "avg v10", 
       avg(ssr) as "avg solar irradiance",
       (180 + 180/pi()*atan2("avg u10", "avg v10"))%360 as "avg wind direction",
       sqrt(power("avg u10", 2) + power("avg v10", 2)) as "avg wind speed",
       --avg(100 * (exp((17.27 * d2m) / (237.3 + d2m)) / exp((17.27 * t2m) / (237.3 + t2m)))) as "humidity"
group by all;

Count


In [43]:
# Visualize the data */
%sql copy (select st_point(longitude, latitude), "avg temperature" from Weather where month = 2 and hour = 6) to 'maps/2. Weather.geojson' with (format GDAL, driver 'geojson');


Count


In [44]:
%%sql /* Query #3 Combine the weather and the spans */
create or replace view spansWithWeather as
from spans, weather
select
       round(least(abs(("avg wind direction" % 180) - (direction % 180)), 180 - abs(("avg wind direction" % 180) - (direction % 180))), 1) AS "line of attack",
       * exclude("avg wind direction","direction", "avg u10", "avg v10", latitude, longitude, "weather longitude", "weather latitude")
where weather.latitude = spans."weather latitude" and weather.longitude = spans."weather longitude";

select "line name", index, "avg temperature", "avg solar irradiance" from spansWithWeather limit 5;

line name,index,avg temperature,avg solar irradiance
MGA-TS 81,227,32.17384382502945,2241762.109865357
MGA-TS 81,227,31.681598220098238,1757186.125443817
MGA-TS 81,227,24.684651752492016,0.0
MGA-TS 81,227,20.900562363913537,0.0
MGA-TS 81,227,33.749054042058106,2736858.180616557


In [22]:
%%sql /* Create a map indexed by time */
copy (
       select span as geometry, concat('2000-01-01 ', hour, ':00:00' )::Datetime as time, * exclude(span, hour) from spansWithWeather
       where month = 1 and 
       ("line name" = 'MOR-TS 81' or "line name" = 'CGT-YLN X1')
) to 'Maps/3. Transmissions with Weather all lines.geojson' (format gdal, driver 'geojson');


Count


In [32]:
import duckdb
from duckdb.typing import *

def line_rating(ambient_temp, wind_speed, angle_of_attack, solar_irradiation, conductor_temp):
    return 0 # not yet implemented

conn = duckdb.connect(':memory:transmission')

conn.create_function("line_rating", line_rating, [DOUBLE,DOUBLE,DOUBLE,DOUBLE,DOUBLE], DOUBLE)

#conn.sql('''from spansWithWeather select * limit 5''')

conn.sql('''from spansWithWeather select "line name", index,
       line_rating("avg temperature", "avg wind speed", "line Of Attack", "avg solar irradiance", 75) as "line rating" limit 5''')


┌───────────┬───────┬─────────────┐
│ line name │ index │ line rating │
│  varchar  │ int64 │   double    │
├───────────┼───────┼─────────────┤
│ TS-MBA 81 │   310 │         0.0 │
│ TS-MBA 81 │   310 │         0.0 │
│ TS-MBA 81 │   310 │         0.0 │
│ TS-MBA 81 │   310 │         0.0 │
│ TS-MBA 81 │   310 │         0.0 │
└───────────┴───────┴─────────────┘

In [ ]:
%%sql
from spansWithWeather
select "line name", 
       line_rating(temperature, "wind speed", "line Of Attack", "solar irradiance", 75) as "line rating" 
